## Starter Notebook


In [6]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

path = "BalancesApril2026//BalancesApril2026//AL-Energy-balance-sheets-April2026-edition.xlsb"   # your downloaded file
xl = pd.ExcelFile(path)
print(xl.sheet_names)

raw = xl.parse(xl.sheet_names[1], header=None)
raw

['Cover', '2024', '2023', '2022', '2021', '2020', '2019', '2018', '2017', '2016', '2015', '2014', '2013', '2012', '2011', '2010', '2009', '2008', '2007', '2006', '2005', '2004', '2003', '2002', '2001', '2000', '1999', '1998', '1997', '1996', '1995', '1994', '1993', '1992', '1991', '1990']


,0,1,2,3,4,5,6,7,8,9,...,70,71,72,73,74,75,76,77,78,79
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TOTAL,C0000X0350-0370,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
4,NaN,NaN,NaN,ktoe,NaN,2024.0,NaN,NaN,Total,Solid fossil fuels,...,Other liquid biofuels,Ambient heat (heat pumps),Non-renewable waste,Industrial waste (non-renewable),Non-renewable municipal waste,Nuclear heat,Heat,Electricity,Fossil energy,Bioenergy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,+,Gross heat production,NaN,NaN,NaN,NaN,NaN,GHP,0,0,...,0,0,0,0,0,0,Z,0,0,0
142,NaN,+,Main activity producer CHP,NaN,NaN,NaN,NaN,GHP_MAPCHP,0,0,...,0,0,0,0,0,0,Z,0,0,0
143,NaN,+,Main activity producer heat only,NaN,NaN,NaN,NaN,GHP_MAPH,0,0,...,0,0,0,0,0,0,Z,0,0,0
144,NaN,+,Autoproducer CHP,NaN,NaN,NaN,NaN,GHP_APCHP,0,0,...,0,0,0,0,0,0,Z,0,0,0


In [7]:
import pandas as pd, numpy as np


xl = pd.ExcelFile(path)

def parse_sheet(name):
    raw = xl.parse(name, header=None)
    country = raw.iloc[1, 0]
    unit    = raw.iloc[4, 3]
    year    = raw.iloc[4, 5]

    codes  = raw.iloc[3, 8:].values
    labels = raw.iloc[4, 8:].values

    body = raw.iloc[5:].copy()
    body = body[body[7].notna()]

    def flow_label(r):
        for c in (0, 1, 2):
            v = r[c]
            if isinstance(v, str) and v not in ("+", "-", "="):
                return v
        return None

    flow_names = body.apply(flow_label, axis=1)
    vals = body.iloc[:, 8:].copy()
    vals.columns = codes
    vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan})
    vals = vals.apply(pd.to_numeric, errors="coerce")
    vals.insert(0, "flow", flow_names.values)
    vals.insert(0, "flow_code", body[7].values)

    out = vals.melt(id_vars=["flow_code", "flow"], var_name="siec", value_name="value")
    out["country"], out["unit"], out["year"] = country, unit, year
    return out

frames = []
for s in xl.sheet_names:
    try:
        f = parse_sheet(s)
        if pd.notna(f["country"].iloc[0]) and pd.notna(f["year"].iloc[0]):
            frames.append(f)
    except Exception as e:
        print("skip", s, "-", type(e).__name__, e)

df = pd.concat(frames, ignore_index=True)
df["year"] = df["year"].astype(int)
print(df.shape, df.country.nunique(), df.year.nunique())
df.head(10)

skip Cover - IndexError index 5 is out of bounds for axis 0 with size 4
(347760, 7) 1 35


,flow_code,flow,siec,value,country,unit,year
0,PPRD,Primary production,TOTAL,1682.411,Albania,ktoe,2024
1,RCV_RCY,Recovered & recycled products,TOTAL,0.000,Albania,ktoe,2024
2,IMP,Imports,TOTAL,1603.810,Albania,ktoe,2024
3,EXP,Exports,TOTAL,990.979,Albania,ktoe,2024
4,STK_CHG,Change in stock,TOTAL,-37.014,Albania,ktoe,2024
5,GAE,Gross available energy,TOTAL,2258.228,Albania,ktoe,2024
6,INTMARB,International maritime bunkers,TOTAL,22.261,Albania,ktoe,2024
7,GIC,Gross inland consumption,TOTAL,2235.967,Albania,ktoe,2024
8,INTAVI,International aviation,TOTAL,87.904,Albania,ktoe,2024
9,NRGSUP,Total energy supply,TOTAL,2148.062,Albania,ktoe,2024


In [4]:
tot = df[df.siec == "TOTAL"]
w = tot.pivot_table(index=["country", "year"], columns="flow_code", values="value").reset_index()

w["import_dep"] = (w["IMP"] - w["EXP"]) / w["GIC"]
w["prod_share"] = w["PPRD"] / w["GIC"]
w[["GIC", "PPRD", "IMP", "EXP", "import_dep"]].describe()

flow_code,GIC,PPRD,IMP,EXP,import_dep
count,35.000000,35.000000,35.000000,35.000000,35.000000
mean,2014.896286,1476.589171,1050.742943,461.923086,0.277130
std,379.739189,398.354825,592.616237,436.560612,0.176409
min,1211.800000,882.998000,34.996000,5.933000,-0.034955
25%,1859.056000,1139.707000,561.283000,66.975000,0.141681
50%,2171.270000,1404.094000,1274.661000,244.107000,0.288968
75%,2248.003500,1767.934500,1496.352500,887.735500,0.436531
max,2672.987000,2459.993000,1920.474000,1246.949000,0.533600


In [9]:
def parse_sheet_wide(name):
    raw = xl.parse(name, header=None)
    country, unit, year = raw.iloc[1, 0], raw.iloc[4, 3], raw.iloc[4, 5]

    codes = raw.iloc[3, 8:].tolist()
    prod_names = dict(zip(codes, raw.iloc[4, 8:].tolist()))

    body = raw.iloc[5:].copy()
    body = body[body[7].notna()].reset_index(drop=True)

    sign  = body[0].where(body[0].isin(["+", "-", "="]))
    sign  = sign.fillna(body[1].where(body[1].isin(["+", "-", "="])))
    level = body[[0, 1, 2]].apply(
        lambda r: next((i for i, v in enumerate(r) if isinstance(v, str) and v not in ("+", "-", "=")), None), axis=1)
    label = body[[0, 1, 2]].apply(
        lambda r: next((v for v in r if isinstance(v, str) and v not in ("+", "-", "=")), None), axis=1)

    vals = body.iloc[:, 8:].copy()
    vals.columns = codes
    vals = vals.replace({"Z": 0, ":": np.nan, "C": np.nan, "X": np.nan}).apply(pd.to_numeric, errors="coerce")

    meta = pd.DataFrame({"country": country, "year": year, "unit": unit,
                         "flow_code": body[7], "flow": label, "sign": sign, "level": level})
    return pd.concat([meta, vals], axis=1), prod_names

wide_frames, prod_names = [], {}
for s in xl.sheet_names:
    try:
        f, p = parse_sheet_wide(s)
        if pd.notna(f["country"].iloc[0]):
            wide_frames.append(f); prod_names.update(p)
    except Exception as e:
        print("skip", s, "-", e)

wide_all = pd.concat(wide_frames, ignore_index=True)
wide_all["year"] = wide_all["year"].astype(int)
#print(wide_all.shape)
wide_all
data = wide_all

skip Cover - index 5 is out of bounds for axis 0 with size 4


In [28]:
twenifow = data[data["year"] == 2024]
twenifow


,country,year,unit,flow_code,flow,sign,level,TOTAL,C0000X0350-0370,C0110,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
0,Albania,2024,ktoe,PPRD,Primary production,+,1,1682.411,119.128,0.000,...,0.0,39.611,0.0,0.0,0.0,0.0,0.0,0.000,781.484,135.623
1,Albania,2024,ktoe,RCV_RCY,Recovered & recycled products,+,1,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
2,Albania,2024,ktoe,IMP,Imports,+,1,1603.810,124.821,124.821,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,229.473,1237.859,136.478
3,Albania,2024,ktoe,EXP,Exports,-,1,990.979,71.429,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,200.763,788.907,1.308
4,Albania,2024,ktoe,STK_CHG,Change in stock,+,1,-37.014,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,-37.014,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,Albania,2024,ktoe,GHP,Gross heat production,+,1,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
134,Albania,2024,ktoe,GHP_MAPCHP,Main activity producer CHP,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
135,Albania,2024,ktoe,GHP_MAPH,Main activity producer heat only,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
136,Albania,2024,ktoe,GHP_APCHP,Autoproducer CHP,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000


In [29]:
twenifre = data[data["year"] == 2023]
twenifre

,country,year,unit,flow_code,flow,sign,level,TOTAL,C0000X0350-0370,C0110,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
138,Albania,2023,ktoe,PPRD,Primary production,+,1,1799.498,188.008,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,867.563,142.359
139,Albania,2023,ktoe,RCV_RCY,Recovered & recycled products,+,1,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.000
140,Albania,2023,ktoe,IMP,Imports,+,1,1395.377,75.409,75.409,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,165.24,1083.801,146.336
141,Albania,2023,ktoe,EXP,Exports,-,1,907.300,119.088,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,244.33,661.886,1.084
142,Albania,2023,ktoe,STK_CHG,Change in stock,+,1,-53.313,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,-53.313,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271,Albania,2023,ktoe,GHP,Gross heat production,+,1,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.000
272,Albania,2023,ktoe,GHP_MAPCHP,Main activity producer CHP,+,2,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.000
273,Albania,2023,ktoe,GHP_MAPH,Main activity producer heat only,+,2,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.000
274,Albania,2023,ktoe,GHP_APCHP,Autoproducer CHP,+,2,0.000,0.000,0.000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.000,0.000


## Spored godina

In [82]:
framesByYear = []
for year in range (1990, 2025):
    framesByYear.append(data[data["year"] == year])
framesByYear[34]

,country,year,unit,flow_code,flow,sign,level,TOTAL,C0000X0350-0370,C0110,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
0,Albania,2024,ktoe,PPRD,Primary production,+,1,1682.411,119.128,0.000,...,0.0,39.611,0.0,0.0,0.0,0.0,0.0,0.000,781.484,135.623
1,Albania,2024,ktoe,RCV_RCY,Recovered & recycled products,+,1,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
2,Albania,2024,ktoe,IMP,Imports,+,1,1603.810,124.821,124.821,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,229.473,1237.859,136.478
3,Albania,2024,ktoe,EXP,Exports,-,1,990.979,71.429,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,200.763,788.907,1.308
4,Albania,2024,ktoe,STK_CHG,Change in stock,+,1,-37.014,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,-37.014,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,Albania,2024,ktoe,GHP,Gross heat production,+,1,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
134,Albania,2024,ktoe,GHP_MAPCHP,Main activity producer CHP,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
135,Albania,2024,ktoe,GHP_MAPH,Main activity producer heat only,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000
136,Albania,2024,ktoe,GHP_APCHP,Autoproducer CHP,+,2,0.000,0.000,0.000,...,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.000,0.000


### Dictonary for the various labels (key, value) i.e. key is feature name, value is dataframe consisting of examples by year i.e. time series for corresponding label 

In [88]:

vals = np.array(data["flow_code"])


print(vals)

framesByFlowCode = {}

for val in vals:
    framesByFlowCode[val] = data[data["flow_code"] == val] 


['PPRD' 'RCV_RCY' 'IMP' ... 'GHP_MAPH' 'GHP_APCHP' 'GHP_APH']


In [89]:
framesByFlowCode

{'PPRD':       country  year  unit flow_code                flow sign  level     TOTAL  \
 0     Albania  2024  ktoe      PPRD  Primary production    +      1  1682.411   
 138   Albania  2023  ktoe      PPRD  Primary production    +      1  1799.498   
 276   Albania  2022  ktoe      PPRD  Primary production    +      1  1607.210   
 414   Albania  2021  ktoe      PPRD  Primary production    +      1  1822.853   
 552   Albania  2020  ktoe      PPRD  Primary production    +      1  1494.103   
 690   Albania  2019  ktoe      PPRD  Primary production    +      1  1736.371   
 828   Albania  2018  ktoe      PPRD  Primary production    +      1  2014.969   
 966   Albania  2017  ktoe      PPRD  Primary production    +      1  1642.327   
 1104  Albania  2016  ktoe      PPRD  Primary production    +      1  1960.857   
 1242  Albania  2015  ktoe      PPRD  Primary production    +      1  2073.399   
 1380  Albania  2014  ktoe      PPRD  Primary production    +      1  2013.445   
 1518  A

In [93]:
framesByFlowCode[vals[2]]

,country,year,unit,flow_code,flow,sign,level,TOTAL,C0000X0350-0370,C0110,...,R5290,RA600,W6100_6220,W6100,W6220,N900H,H8000,E7000,FE,BIOE
2,Albania,2024,ktoe,IMP,Imports,+,1,1603.810,124.821,124.821,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,229.473,1237.859,136.478
140,Albania,2023,ktoe,IMP,Imports,+,1,1395.377,75.409,75.409,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,165.240,1083.801,146.336
278,Albania,2022,ktoe,IMP,Imports,+,1,1534.899,51.784,51.784,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,261.697,1136.130,137.073
416,Albania,2021,ktoe,IMP,Imports,+,1,1466.358,95.966,95.966,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,193.637,1157.852,114.869
554,Albania,2020,ktoe,IMP,Imports,+,1,1608.852,81.261,81.261,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,278.472,1214.309,116.071
692,Albania,2019,ktoe,IMP,Imports,+,1,1634.871,71.561,71.561,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,273.131,1233.783,127.957
830,Albania,2018,ktoe,IMP,Imports,+,1,1396.228,78.032,78.032,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,73.128,1197.181,125.919
968,Albania,2017,ktoe,IMP,Imports,+,1,1572.824,47.569,47.569,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,250.613,1228.637,93.574
1106,Albania,2016,ktoe,IMP,Imports,+,1,1374.527,47.894,47.894,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000,1289.430,85.097
1244,Albania,2015,ktoe,IMP,Imports,+,1,1526.347,60.311,60.311,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,202.494,1292.219,31.634


In [94]:
import eurostat
df = eurostat.get_data_df("nrg_bal_c")

In [106]:
sector_codes = np.array(df["nrg_bal"].unique())
df

,freq,nrg_bal,siec,unit,geo,1990,1991,1992,1993,1994,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,A,AFC,BIOE,GWH,AL,4221.944,4221.944,4221.944,4029.722,3846.389,...,2728.383,3049.733,2804.720,3129.617,3211.056,3013.181,2963.553,3180.853,3291.735,3098.084
1,A,AFC,BIOE,GWH,AT,25744.633,27905.111,27110.141,27367.890,25186.310,...,57495.297,57863.539,58217.651,55736.679,55563.334,54830.084,60953.408,59869.408,62017.419,62384.826
2,A,AFC,BIOE,GWH,BA,NaN,NaN,NaN,NaN,NaN,...,5937.175,5936.294,4843.579,13379.486,13990.576,14750.870,14124.598,15190.525,14556.421,14291.273
3,A,AFC,BIOE,GWH,BE,4034.103,4108.185,3958.904,3139.375,3076.269,...,25304.247,27008.692,27175.362,27777.464,25898.526,27773.762,30739.630,30618.106,30145.021,29175.304
4,A,AFC,BIOE,GWH,BG,2005.833,1333.333,1818.057,1656.454,1865.723,...,13750.594,14423.227,14656.419,16327.377,16722.072,18123.927,17830.582,16818.733,15058.769,14226.252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216492,A,TO_RPI_RO,W6220,TJ,SK,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216493,A,TO_RPI_RO,W6220,TJ,TR,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216494,A,TO_RPI_RO,W6220,TJ,UA,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1216495,A,TO_RPI_RO,W6220,TJ,UK,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Reshaping

In [105]:
df = df.rename(columns={r"geo\TIME_PERIOD": "geo"})
years = [c for c in df.columns if str(c).isdigit()]

long = df.melt(id_vars=["freq", "nrg_bal", "siec", "unit", "geo"],
               value_vars=years, var_name="year", value_name="value")
long["year"] = long["year"].astype(int)
long = long.dropna(subset=["value"])
print(long.shape)
long.head()

(21300267, 7)


,freq,nrg_bal,siec,unit,geo,year,value
0,A,AFC,BIOE,GWH,AL,1990,4221.944
1,A,AFC,BIOE,GWH,AT,1990,25744.633
3,A,AFC,BIOE,GWH,BE,1990,4034.103
4,A,AFC,BIOE,GWH,BG,1990,2005.833
5,A,AFC,BIOE,GWH,CY,1990,48.472


## Filter for Albania (just as an example)

In [107]:
al = long[long.geo == "AL"].copy()
print(al.country if "country" in al else al.geo.unique(), al.year.min(), al.year.max())

<ArrowStringArray>
['AL']
Length: 1, dtype: str 1990 2024


In [ ]:
# By flow: one row per year, one column per fuel, for a given flow
def by_flow(flow_code, country="AL", unit="KTOE"):
    d = long[(long.nrg_bal == flow_code) & (long.geo == country) & (long.unit == unit)]
    return d.pivot(index="year", columns="siec", values="value").sort_index()

gic = by_flow("GIC")
pprd = by_flow("PPRD")

# Year × flow for TOTAL fuel
def year_by_flow(country="AL", unit="KTOE", fuel="TOTAL"):
    d = long[(long.geo == country) & (long.unit == unit) & (long.siec == fuel)]
    return d.pivot(index="year", columns="nrg_bal", values="value").sort_index()

yf = year_by_flow()
print(yf[["PPRD", "IMP", "EXP", "GIC"]].head())

nrg_bal      PPRD       IMP       EXP       GIC
year                                           
1990     2459.993   292.185   120.792  2672.987
1991     1923.294   129.999   195.397  1870.895
1992     1404.094    48.407    94.286  1358.214
1993     1332.201    34.996    32.416  1331.116
1994     1283.281    85.899    35.107  1416.420
1995     1241.200    95.443     6.363  1330.280
1996     1329.514   122.885    32.674  1419.725
1997     1105.456   178.485    72.141  1211.800
1998     1089.078   277.781    59.329  1336.329
1999     1070.235   830.381    64.918  1843.378
2000      986.179   866.773    21.095  1847.217
2001      882.998   990.537     5.933  1885.842
2002      945.602  1274.661   163.159  2083.024
2003     1004.471  1156.491   121.151  2039.811
2004     1169.259  1138.193    69.032  2238.419
2005     1131.883  1164.435    45.619  2250.699
2006     1218.870   990.592   109.225  2182.327
2007     1050.021  1255.494   244.107  2056.525
2008     1147.531  1319.671   254.355  2

## Random analyses

In [112]:
# Trend test
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
tau, p = stats.kendalltau(yf.index, yf["GIC"])
print("GIC trend tau:", round(tau, 2), "p:", round(p, 4))

# Correlation
print(yf[["PPRD", "IMP", "EXP", "GIC"]].corr(method="spearman").round(2))

# Regression
X = sm.add_constant(yf[["PPRD", "IMP"]])
m = sm.OLS(yf["GIC"], X).fit()
print(m.summary())

GIC trend tau: 0.52 p: 0.0
nrg_bal  PPRD   IMP   EXP   GIC
nrg_bal                        
PPRD     1.00  0.47  0.73  0.62
IMP      0.47  1.00  0.84  0.76
EXP      0.73  0.84  1.00  0.66
GIC      0.62  0.76  0.66  1.00
                            OLS Regression Results                            
Dep. Variable:                    GIC   R-squared:                       0.745
Model:                            OLS   Adj. R-squared:                  0.730
Method:                 Least Squares   F-statistic:                     46.87
Date:                Tue, 20 Oct 2026   Prob (F-statistic):           3.10e-10
Time:                        19:06:17   Log-Likelihood:                -233.09
No. Observations:                  35   AIC:                             472.2
Df Residuals:                      32   BIC:                             476.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                 

In [113]:
print(sorted(long.nrg_bal.unique())[:20])
print(sorted(long.siec.unique())[:20])

['AFC', 'DL', 'EXP', 'FC_E', 'FC_IND_CON_E', 'FC_IND_CPC_E', 'FC_IND_E', 'FC_IND_FBT_E', 'FC_IND_IS_E', 'FC_IND_MAC_E', 'FC_IND_MQ_E', 'FC_IND_NE', 'FC_IND_NFM_E', 'FC_IND_NMM_E', 'FC_IND_NSP_E', 'FC_IND_PPP_E', 'FC_IND_TE_E', 'FC_IND_TL_E', 'FC_IND_WP_E', 'FC_NE']
['BIOE', 'C0000X0350-0370', 'C0110', 'C0121', 'C0129', 'C0210', 'C0220', 'C0311', 'C0312', 'C0320', 'C0330', 'C0340', 'C0350', 'C0350-0370', 'C0360', 'C0371', 'C0379', 'E7000', 'FE', 'G3000']
